# StereoQueerEval Task B: Pretrained Attention Weight Borrowing with $K$-Latent Queries
### Clean Modular Kaggle Runner (GPU P100 / T4 / A100)

This notebook imports the modular pipeline directly from `src.*`.

**Architecture Specifications:**
- **Context Stream**: `mmBERT-base` (ModernBERT, layers 0..20, native full/sliding-window masks).
- **Query Stream**: $K$ unconstrained learnable latent query slots $Z_0 \in \mathbb{R}^{B \times K \times 768}$.
- **Transplant Stack (Layers 18 $\to$ 22)**: Parameter-decoupled projections with **Asymmetric RoPE** (Keys rotated with native ModernBERT RoPE, Queries position-free).
- **Context Synchronization**: Layer-matched mode ($K_l, V_l$ receive $H_{l-1}$ for $l \in [18..22]$).
- **Hierarchical Tree Classifier**: Attentive readout producing compound probabilities $P(\text{no})$, $P(\text{yes\_implicit})$, $P(\text{yes\_explicit})$ via joint NLL.

## 1. Setup Environment & Clone Repository

In [ ]:
# 1. Install modern transformers with native ModernBERT support
!pip install -q "transformers>=4.48.0" accelerate scikit-learn pandas numpy matplotlib seaborn tqdm

# 2. Clone repository if running on Kaggle
!if [ ! -d "lgbtia" ] && [ ! -d "src" ]; then \
    git clone https://github.com/AmnO-O/lgbtia.git; \
fi

import os
import sys

# Ensure project root is in Python sys.path
for candidate_root in ["./lgbtia", "."]:
    if os.path.exists(os.path.join(candidate_root, "src")):
        if candidate_root not in sys.path:
            sys.path.insert(0, candidate_root)
        print(f"Project root added to sys.path: {candidate_root}")
        break

# Verify GPU device
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Import Modular Pipeline from `src.*`

In [ ]:
# Import directly from modular repository source code
from src.config import ModelConfig, TrainingConfig
from src.train import train_task_b_model
from src.predict import HateSpeechPredictor
from src.metrics import compute_task_b_metrics, evaluate_task_b_by_language

print("All modular components successfully imported from src!")

## 3. Configure Experiment

In [ ]:
# Set model configuration
model_cfg = ModelConfig(
    model_name_or_path="jhu-clsp/mmbert-base",
    num_query_slots=8,            # K general latent queries
    split_layer_idx=17,           # Layers 0..16 -> Context Stream (runs 0..20)
    transplant_layers_count=5,    # Layers 18..22 -> Transplant Stack
    context_mode="layer_matched", # Default: Layer l receives H_{l-1}
    use_prenorm=False,            # False = Exp A, True = Exp B (Pre-LN)
    use_ffn_transplant=False,     # False = Primary (no token-to-query FFN shift)
    freeze_encoder=True,          # Freeze Context Layers 1..17
    freeze_transplant=True,       # Freeze Transplant Projections (Isolated Hypothesis)
    dropout=0.1
)

# Set training hyperparameters
train_cfg = TrainingConfig(
    batch_size=16,
    epochs=10,
    encoder_lr=1e-5,
    transplant_lr=2e-4,
    head_lr=3e-4,
    weight_decay=0.01,
    val_size=0.15,
    max_length=384,
    device=str(device),
    output_dir="./outputs_task_b"
)

os.makedirs(train_cfg.output_dir, exist_ok=True)
print("Model Configuration:", vars(model_cfg))
print("Training Configuration:", vars(train_cfg))

## 4. Run End-to-End Training

In [ ]:
# Train model using the repository's training engine with per-epoch logging
trained_model, results = train_task_b_model(model_cfg=model_cfg, train_cfg=train_cfg)

best_macro_f1 = results.get('best_val_macro_f1', 0.0)
log_history = results.get('history', [])
print(f"\nTraining finished! Peak Validation Macro F1: {best_macro_f1:.4f}")

## 5. Result Analysis & Diagnostic Visualizations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Plot Learning Curves
log_df = pd.DataFrame(log_history)

if not log_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(log_df['epoch'], log_df['train_loss'], marker='o', color='crimson', label='Train Loss')
    ax1.plot(log_df['epoch'], log_df['val_loss'], marker='s', color='navy', linestyle='--', label='Val Loss')
    ax1.set_title('Hierarchical Loss Curves', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()

    ax2.plot(log_df['epoch'], log_df['val_macro_f1'], marker='s', color='forestgreen', label='Val Macro F1')
    ax2.plot(log_df['epoch'], log_df['val_accuracy'], marker='^', color='teal', label='Val Accuracy')
    ax2.plot(log_df['epoch'], log_df['f1_implicit'], marker='.', color='orange', linestyle=':', label='F1 (Implicit)')
    ax2.set_title('Validation Metrics per Epoch', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(train_cfg.output_dir, "learning_curves.png"), dpi=300)
    plt.show()

### Confusion Matrix & Per-Language Breakdown

In [ ]:
final_metrics = results.get('final_metrics', {})
breakdown_df = results.get('language_breakdown', pd.DataFrame())

if 'report' in final_metrics:
    print("\nValidation Classification Report:")
    print(final_metrics['report'])

if not breakdown_df.empty:
    print("\nPer-Language Performance Breakdown:")
    display(breakdown_df)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=breakdown_df, x='language', y='macro_f1', palette='viridis')
    plt.title('Task B Macro F1 Across Languages (IT, NL, EN, FA)', fontweight='bold')
    plt.ylim(0, 1.0)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(train_cfg.output_dir, "language_breakdown.png"), dpi=300)
    plt.show()

## 6. Generate Test Submission TSV

In [ ]:
checkpoint_path = os.path.join(train_cfg.output_dir, "best_model.pt")
predictor = HateSpeechPredictor(checkpoint_path=checkpoint_path, device=str(device))

# Search for official test TSV
test_paths = ["./lgbtia/data/test_task_b.tsv", "./data/test_task_b.tsv", "./test_task_b.tsv"]
found_test_path = next((p for p in test_paths if os.path.exists(p)), None)

if found_test_path:
    print(f"Generating predictions for {found_test_path}...")
    test_df = pd.read_csv(found_test_path, sep='\t')
    preds, probs = predictor.predict_batch(
        comments=test_df['comment'].tolist(),
        titles=test_df.get('yt_title', pd.Series(['']*len(test_df))).tolist(),
        descriptions=test_df.get('yt_description', pd.Series(['']*len(test_df))).tolist()
    )
    test_df['label'] = preds
    sub_path = os.path.join(train_cfg.output_dir, "submission_task_b.tsv")
    test_df[['id', 'label']].to_csv(sub_path, sep='\t', index=False)
    print(f"Submission successfully saved to {sub_path} ({len(test_df)} rows).")
    display(test_df[['id', 'label']].head(10))
else:
    print("No test_task_b.tsv found. Interactive demo prediction:")
    sample_res = predictor.predict_batch(
        comments=["This video is fantastic and educational!", "These people should not have rights in our country."],
        titles=["LGBTQ+ Community Discussion", "Pride Parade 2024"],
        descriptions=["A discussion about rights and social acceptance", "Coverage of the event"]
    )
    print("Demo Predictions:", sample_res[0])